<a href="https://colab.research.google.com/github/adrianwedd/afterwords/blob/main/colab/afterwords_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Afterwords — Qwen3-TTS Model Comparison

Compare the **0.6B** and **1.7B** Qwen3-TTS models side by side.
Clone any voice from a reference clip and hear the difference.

**What this notebook does:**
1. Loads each model one at a time (fits on free Colab T4)
2. Clones voices from reference audio + transcript
3. Plays results inline so you can compare quality
4. Bonus: 1.7B VoiceDesign mode — describe a voice in words, no audio needed

**Requirements:** GPU runtime (T4 is fine). Go to *Runtime → Change runtime type → T4 GPU*.

---

## 1. Setup

Install dependencies and verify GPU access.

In [1]:
# Install qwen-tts, audio libraries, and optimizations
!pip install -q qwen-tts soundfile IPython flash-attn

# Install SoX to handle soundfile warnings and improve audio processing robustness
!apt-get update && apt-get install -y sox libsox-dev libsox-fmt-all

import torch
import gc

# Verify GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU."
    )

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu} ({vram:.1f} GB VRAM)")
print("Ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.0 MB/s eta 0:00:00
GPU: Tesla T4 (15.6 GB VRAM)
Ready.


## 2. Voice References

Choose your reference voice. You can either:
- **Use a shipped voice** from the Afterwords repo (cloned below)
- **Upload your own** WAV file

The reference audio should be ~15 seconds of clean speech with an accurate transcript.

In [14]:
import os
import json
import ipywidgets as widgets
from IPython.display import display, Audio, HTML
from google.colab import files

# Clone the repo to get shipped voice profiles
if not os.path.exists("afterwords"):
    !git clone --depth 1 https://github.com/adrianwedd/afterwords.git

VOICES_DIR = "afterwords/voices"


def load_voice_profiles():
    """Load all voice profiles from the repo."""
    profiles = {}
    for f in sorted(os.listdir(VOICES_DIR)):
        if not f.endswith(".json"):
            continue
        path = os.path.join(VOICES_DIR, f)
        with open(path) as fh:
            p = json.load(fh)
        name = p.get("name", f.replace(".json", ""))
        ref_wav = os.path.join(VOICES_DIR, p["reference_audio"])
        if os.path.exists(ref_wav) and p.get("reference_text"):
            profiles[name] = {
                "audio": ref_wav,
                "text": p["reference_text"],
            }
    return profiles


profiles = load_voice_profiles()
print(f"Loaded {len(profiles)} voice profiles:")
for name in profiles:
    print(f"  - {name}")

Loaded 4 voice profiles:
  - bardem
  - data
  - picard
  - ronan


In [5]:
# ── Choose a voice ──────────────────────────────────────────────
#
# Option A: Pick a shipped voice (edit the name below)
VOICE_NAME = "data"

# Option B: Upload your own (uncomment these lines)
# uploaded = files.upload()  # upload a WAV file
# VOICE_NAME = "custom"
# profiles["custom"] = {
#     "audio": list(uploaded.keys())[0],
#     "text": "Type the exact transcript of your audio here.",
# }

# ── Text to synthesize ─────────────────────────────────────────
SYNTH_TEXT = (
    "You are absolutely right. Your Claude Code session "
    "could sound like me."
)

# ────────────────────────────────────────────────────────────────
voice = profiles[VOICE_NAME]
print(f"Voice: {VOICE_NAME}")
print(f"Reference: {voice['audio']}")
print(f"Transcript: {voice['text'][:80]}...")
print(f"\nText to synthesize: {SYNTH_TEXT}")
print("\nReference audio:")
display(Audio(voice["audio"]))

Voice: data
Reference: afterwords/voices/data-ref.wav
Transcript: I've never eaten before. What do I ask for? The choice of meal is determined by ...

Text to synthesize: You are absolutely right. Your Claude Code session could sound like me.

Reference audio:


## 3. Generate with 0.6B Model

The smaller model — same one Afterwords uses locally on Apple Silicon (but here in FP16 on CUDA, not 8-bit MLX).

In [6]:
import time
import soundfile as sf
from qwen_tts import Qwen3TTSModel


def load_model(model_id):
    """Load a Qwen3-TTS model, clearing any previous model from GPU."""
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Loading {model_id}...")
    t0 = time.time()
    model = Qwen3TTSModel.from_pretrained(
        model_id,
        device_map="cuda:0",
        dtype=torch.bfloat16,
    )
    elapsed = time.time() - t0
    vram_used = torch.cuda.memory_allocated() / 1e9
    print(f"Loaded in {elapsed:.1f}s — VRAM: {vram_used:.1f} GB")
    return model


def clone_voice(model, text, ref_audio, ref_text):
    """Generate speech by cloning a voice."""
    t0 = time.time()
    wavs, sr = model.generate_voice_clone(
        text=text,
        language="English",
        ref_audio=ref_audio,
        ref_text=ref_text,
        max_new_tokens=2048,
        temperature=0.9,
        top_p=1.0,
        top_k=50,
        repetition_penalty=1.05,
    )
    elapsed = time.time() - t0
    duration = len(wavs[0]) / sr
    print(f"Generated {duration:.1f}s audio in {elapsed:.1f}s")
    return wavs[0], sr


# Load 0.6B
model_06b = load_model("Qwen/Qwen3-TTS-12Hz-0.6B-Base")

# Generate
print(f"\nSynthesizing with 0.6B...")
wav_06b, sr_06b = clone_voice(
    model_06b, SYNTH_TEXT, voice["audio"], voice["text"]
)
sf.write("output_06b.wav", wav_06b, sr_06b)

# Free GPU memory
del model_06b
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM freed: {torch.cuda.memory_allocated() / 1e9:.1f} GB in use")

# Play
print("\n0.6B result:")
display(Audio("output_06b.wav"))


    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 
Loading Qwen/Qwen3-TTS-12Hz-0.6B-Base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.83G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Loaded in 39.5s — VRAM: 2.2 GB

Synthesizing with 0.6B...


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Generated 3.8s audio in 24.7s
VRAM freed: 0.0 GB in use

0.6B result:


## 4. Generate with 1.7B Model

The larger model — nearly 3x the parameters. Should produce more natural prosody and better voice matching.

In [7]:
# Load 1.7B
model_17b = load_model("Qwen/Qwen3-TTS-12Hz-1.7B-Base")

# Generate
print(f"\nSynthesizing with 1.7B...")
wav_17b, sr_17b = clone_voice(
    model_17b, SYNTH_TEXT, voice["audio"], voice["text"]
)
sf.write("output_17b.wav", wav_17b, sr_17b)

# Free GPU memory
del model_17b
gc.collect()
torch.cuda.empty_cache()

# Play
print("\n1.7B result:")
display(Audio("output_17b.wav"))

Loading Qwen/Qwen3-TTS-12Hz-1.7B-Base...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Loaded in 40.2s — VRAM: 4.2 GB

Synthesizing with 1.7B...


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Generated 3.6s audio in 11.4s

1.7B result:


## 5. Side-by-Side Comparison

Listen to both outputs together.

In [8]:
display(HTML(f"""
<h3>Voice: {VOICE_NAME}</h3>
<p><em>"{SYNTH_TEXT}"</em></p>
<table style="width:100%; border-collapse: collapse;">
<tr style="background: #1a1a2e; color: #e0e0e0;">
  <th style="padding: 12px; text-align: left; width: 50%;">0.6B (local default)</th>
  <th style="padding: 12px; text-align: left; width: 50%;">1.7B (larger model)</th>
</tr>
</table>
"""))

print("0.6B:")
display(Audio("output_06b.wav"))
print("\n1.7B:")
display(Audio("output_17b.wav"))

0.6B (local default),1.7B (larger model)


0.6B:



1.7B:


## 6. VoiceDesign Mode (1.7B only)

This is the unique capability of the 1.7B model — **describe the voice you want in plain English** and it generates speech without any reference audio.

Try different descriptions to explore what's possible.

In [9]:
# Load VoiceDesign model
model_vd = load_model("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign")

# ── Edit these to experiment ───────────────────────────────────
VOICE_DESCRIPTION = (
    "Speak in a deep, theatrical British voice with a hint of "
    "mischief, like a Shakespearean villain who finds everything "
    "secretly amusing."
)
DESIGN_TEXT = (
    "I must say, your code is quite remarkable. "
    "Almost as remarkable as mine."
)
# ──────────────────────────────────────────────────────────────

print(f"Voice description: {VOICE_DESCRIPTION}")
print(f"Text: {DESIGN_TEXT}\n")

t0 = time.time()
wavs, sr = model_vd.generate_voice_design(
    text=DESIGN_TEXT,
    language="English",
    instruct=VOICE_DESCRIPTION,
    max_new_tokens=2048,
)
elapsed = time.time() - t0
duration = len(wavs[0]) / sr
print(f"Generated {duration:.1f}s audio in {elapsed:.1f}s")
sf.write("output_voice_design.wav", wavs[0], sr)

display(Audio("output_voice_design.wav"))

Loading Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Loaded in 85.5s — VRAM: 4.2 GB
Voice description: Speak in a deep, theatrical British voice with a hint of mischief, like a Shakespearean villain who finds everything secretly amusing.
Text: I must say, your code is quite remarkable. Almost as remarkable as mine.

Generated 6.6s audio in 18.8s


In [13]:
# ── Try more voice descriptions ────────────────────────────────
#
# Uncomment any of these and re-run the cell above:
#
VOICE_DESCRIPTION = "A warm, gentle grandmother reading a bedtime story."
VOICE_DESCRIPTION = "Speak in an incredulous tone, with panic creeping into your voice."
VOICE_DESCRIPTION = "A calm, measured newsreader with a slight Australian accent."
VOICE_DESCRIPTION = "An excited sports commentator calling a last-second goal."
VOICE_DESCRIPTION = "A tired, world-weary detective narrating a case."

# Free GPU when done
del model_vd
gc.collect()
torch.cuda.empty_cache()
print("VoiceDesign model freed.")

NameError: name 'model_vd' is not defined

## 7. Batch Comparison (All Shipped Voices)

Run the same phrase through **every shipped voice** with both models. This takes a while but gives a comprehensive quality comparison.

Skip this cell if you just wanted a quick test.

In [ ]:
import os

BATCH_TEXT = "You are absolutely right. Your Claude Code session could sound like me."
os.makedirs("comparison", exist_ok=True)

results = {}

for model_label, model_id in [
    ("0.6B", "Qwen/Qwen3-TTS-12Hz-0.6B-Base"),
    ("1.7B", "Qwen/Qwen3-TTS-12Hz-1.7B-Base"),
]:
    model = load_model(model_id)
    results[model_label] = {}

    for name, voice in profiles.items():
        print(f"  {model_label} × {name}...", end=" ")
        try:
            wav, sr = clone_voice(
                model, BATCH_TEXT, voice["audio"], voice["text"]
            )
            out_path = f"comparison/{name}_{model_label}.wav"
            sf.write(out_path, wav, sr)
            results[model_label][name] = out_path
        except Exception as e:
            print(f"FAILED: {e}")
            results[model_label][name] = None

    del model
    gc.collect()
    torch.cuda.empty_cache()

print("\nDone. Listen to the results below.")

In [12]:
# Play batch results side by side
for name in profiles:
    path_06b = results.get("0.6B", {}).get(name)
    path_17b = results.get("1.7B", {}).get(name)
    if not path_06b and not path_17b:
        continue

    display(HTML(f"<h4>{name}</h4>"))
    if path_06b:
        print("0.6B:")
        display(Audio(path_06b))
    if path_17b:
        print("1.7B:")
        display(Audio(path_17b))
    print()

NameError: name 'results' is not defined

## 8. Download Results

Download all generated audio as a zip file.

In [ ]:
import shutil
import glob

# Collect all generated files
wavs = glob.glob("output_*.wav") + glob.glob("comparison/*.wav")
if wavs:
    os.makedirs("download", exist_ok=True)
    for w in wavs:
        shutil.copy(w, "download/")
    shutil.make_archive("afterwords_comparison", "zip", "download")
    files.download("afterwords_comparison.zip")
    print(f"Downloaded {len(wavs)} audio files.")
else:
    print("No audio files to download. Run the generation cells first.")

---

### Notes

-   **Local use:** Afterwords runs the 0.6B model (8-bit quantized) on Apple Silicon via MLX. See [github.com/adrianwedd/afterwords](https://github.com/adrianwedd/afterwords).
-   **This notebook** uses the official PyTorch weights in FP16 on CUDA — different backend, same voices.
-   **VoiceDesign** is 1.7B-only. It creates voices from text descriptions — no reference audio needed.
-   All models by [Qwen/Alibaba](https://github.com/QwenLM/Qwen3-TTS), Apache 2.0 license.